# Chess-Coach Agent — **Gemma 4 E4B** QLoRA on Kaggle **2×T4**

Trains a LoRA adapter on the v1.2 agentic-harness SFT corpus (skills + tools + the
`python` verification tool; fast/think/auto modes), then zips the adapter to download
and serve locally as GGUF on the RTX 4060.

## Why 2×T4 (not 1)
E4B 4-bit at **seq 1664** does **not** fit one 16 GB T4 — the forward pass alone OOMs.
With two T4s the trainer auto-sets `device_map="balanced"`: it **shards** the model
across both GPUs (~28 GB usable, model-parallel). This is NOT DDP (DDP replicates the
model per GPU and OOMs). Single T4 → use the E2B notebook instead.

## Kaggle setup (do this first)
1. **Settings → Accelerator → GPU T4 ×2** (required for E4B).
2. **Settings → Internet → On** (clone repo + download base).
3. **Add-ons → Secrets** → add `HF_TOKEN` (HF read token). Accept the Gemma 4 license
   once on the model page: https://huggingface.co/unsloth/gemma-4-E4B-it
4. Run cells top→bottom. After Cell 4 (deps) Kaggle may need a **restart** — restart,
   then continue from Cell 5. **Cell 6.5 (fit test) is the go/no-go** before the long run.

## OOM ladder (pessimistic — never cut seq; it's the data floor)
RANK 16→8 → TARGETS attn-only → (last resort) the E2B notebook. seq stays 1664.


In [ ]:
# Cell 1 — config (E4B QLoRA on Kaggle 2×T4, model-parallel sharded)
import os

MODEL  = "gemma4_e4b"
ENGINE = "unsloth"   # Gemma-4-native: fused kernels + ~70% less VRAM. On 2 GPUs the
                     # trainer auto-sets device_map="balanced" (shard, not DDP).

# Unsloth-published base (load_in_4bit). NOT a raw Google QAT checkpoint — that makes
# Unsloth re-init base k/v weights and train_unsloth.py fails LOUD (BaseReinitError).
HF_REPO = {
    ("gemma4_e4b", "unsloth"): "unsloth/gemma-4-E4B-it",
    ("gemma4_e2b", "unsloth"): "unsloth/gemma-4-E2B-it",
    ("gemma4_e4b", "cuda"):    "google/gemma-4-E4B-it",
    ("gemma4_e2b", "cuda"):    "google/gemma-4-E2B-it",
}[(MODEL, ENGINE)]
NO_4BIT = False  # E4B MUST be 4-bit

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_kaggle"

# SEQ is set by the DATA, not VRAM: corpus max row 1655, MEDIAN 1291. Below 1664 the
# FINAL grounded answer truncates off >50% of rows. FIXED at 1664 — do not lower.
MAX_SEQ = 1664
TARGETS = "attn-only"    # OOM ladder: attn-only + rank 16 -> rank 8 -> E2B. Never cut seq.
RANK = 16
GRAD_ACCUM = 16          # effective batch = 1 × 16; raise freely (no VRAM cost)
MAX_STEPS = 800          # real run on the 30h slot; raise/lower to taste
EVAL_EVERY = 100
MAX_VAL = 128
print("config:", MODEL, "engine=", ENGINE, "base=", HF_REPO, "seq=", MAX_SEQ,
      "rank=", RANK, "targets=", TARGETS, "steps=", MAX_STEPS)


In [ ]:
# Cell 2 — GPU preflight (E4B needs 2×T4 for the sharded fit)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Settings → Accelerator → GPU T4 ×2"
n = torch.cuda.device_count()
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0), "| count", n)
if n < 2:
    print("\n⚠️  Only 1 GPU — E4B@1664 will OOM (forward alone doesn't fit one T4).\n"
          "    Set Accelerator to GPU T4 ×2, or use the E2B notebook on a single T4.")


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps. Let Unsloth resolve its own current stack (do NOT pin an old
# transformers — Gemma 4 needs a recent one).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

if ENGINE == "unsloth":
    pip("--upgrade", "unsloth", "unsloth_zoo", "bitsandbytes")
    pip("python-chess")
    print("unsloth installed. If Kaggle shows a RESTART banner, restart, then run Cells 5+. "
          "If Gemma 4 fails to load with a model-type error, run: pip install -U transformers")
else:
    pip("-U", "transformers", "accelerate", "peft", "trl",
        "bitsandbytes", "datasets", "sentencepiece", "protobuf", "python-chess")
    import transformers, peft, bitsandbytes
    print("transformers", transformers.__version__, "| peft", peft.__version__,
          "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in ("v1_2_train.jsonl", "v1_2_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 7 — train (E4B QLoRA, Unsloth, auto device_map=balanced on 2 GPUs) -> runs/<OUTPUT>
import subprocess, sys, os
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--engine", ENGINE, "--max-steps", str(MAX_STEPS),
       "--rank", str(RANK), "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM),
       "--max-seq", str(MAX_SEQ), "--eval-every", str(EVAL_EVERY), "--max-val", str(MAX_VAL),
       "--output", OUTPUT]
if NO_4BIT:
    cmd.append("--no-4bit")
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})


In [ ]:
# Cell 7 — train QLoRA -> LoRA adapter under runs/<OUTPUT>
import subprocess, sys, os
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--max-steps", str(MAX_STEPS), "--rank", str(RANK),
       "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM), "--max-seq", str(MAX_SEQ),
       "--output", OUTPUT]
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm"})


In [ ]:
# Cell 8 — zip the adapter to /kaggle/working for download
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("adapter files:", os.listdir(run_dir))
out_zip = f"/kaggle/working/{OUTPUT}_adapter"
shutil.make_archive(out_zip, "zip", run_dir)
print("download from the Output panel:", out_zip + ".zip")
